[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/assignments/week03_assignment_early_results.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 3 Assignment: Early Results & Stakeholder Management

## Context

You are a product data scientist at **FamilyNest** (Airbnb for families with young kids). You have a new PM named **Max** (short for "Maximus Impact") who is enthusiastic, eager to ship things fast, and sometimes pushes to make decisions before the data is ready.

This week you will practice:
- Designing experiments and calculating run time
- Analyzing early and intermediate results
- Identifying novelty effects
- Managing stakeholder expectations when results are tempting but inconclusive

Three tests this week all involve careful handling of early/intermediate results and stakeholder management.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def get_ci(series, confidence=0.95):
    """
    Calculate the confidence interval for a series.
    Returns (mean, ci_lower, ci_upper)
    """
    n = len(series)
    mean = series.mean()
    se = series.std() / np.sqrt(n)
    z = stats.norm.ppf((1 + confidence) / 2)
    ci_lower = mean - z * se
    ci_upper = mean + z * se
    return mean, ci_lower, ci_upper

In [ ]:
def calculate_results(df, metric, group_col='test_group', control_label='control', 
                      treatment_label='treatment', alpha=0.05):
    """
    Calculate A/B test results for a binary metric.
    Accepts standard metrics as well as 'bookings_4w'.
    
    Returns a summary with conversion rates, lift, z-test results.
    """
    control = df[df[group_col] == control_label][metric]
    treatment = df[df[group_col] == treatment_label][metric]
    
    n_control = len(control)
    n_treatment = len(treatment)
    
    rate_control = control.mean()
    rate_treatment = treatment.mean()
    
    lift = (rate_treatment - rate_control) / rate_control if rate_control > 0 else 0
    
    # Two-proportion z-test
    successes = np.array([treatment.sum(), control.sum()])
    nobs = np.array([n_treatment, n_control])
    z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')
    
    significant = p_value < alpha
    
    print(f"Metric: {metric}")
    print(f"Control:   n={n_control:,}, rate={rate_control:.4f}")
    print(f"Treatment: n={n_treatment:,}, rate={rate_treatment:.4f}")
    print(f"Lift: {lift:.4f} ({lift*100:.2f}%)")
    print(f"Z-statistic: {z_stat:.4f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Significant at alpha={alpha}: {significant}")
    print("---")
    
    return {
        'metric': metric,
        'n_control': n_control,
        'n_treatment': n_treatment,
        'rate_control': rate_control,
        'rate_treatment': rate_treatment,
        'lift': lift,
        'z_stat': z_stat,
        'p_value': p_value,
        'significant': significant
    }

In [ ]:
def custom_calculate_results(df, metric, group_col='test_group', control_label='control', 
                             treatment_label='treatment', alpha=0.05):
    """
    Calculate A/B test results and return as a DataFrame row.
    Useful for building results tables across multiple metrics/dimensions.
    """
    control = df[df[group_col] == control_label][metric]
    treatment = df[df[group_col] == treatment_label][metric]
    
    n_control = len(control)
    n_treatment = len(treatment)
    
    rate_control = control.mean()
    rate_treatment = treatment.mean()
    
    lift = (rate_treatment - rate_control) / rate_control if rate_control > 0 else 0
    
    # Two-proportion z-test
    successes = np.array([treatment.sum(), control.sum()])
    nobs = np.array([n_treatment, n_control])
    z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')
    
    significant = p_value < alpha
    
    result = pd.DataFrame([{
        'metric': metric,
        'n_control': n_control,
        'n_treatment': n_treatment,
        'rate_control': rate_control,
        'rate_treatment': rate_treatment,
        'lift': lift,
        'lift_pct': f"{lift*100:.2f}%",
        'z_stat': z_stat,
        'p_value': p_value,
        'significant': significant
    }])
    
    return result

In [ ]:
def ab_test_chi2(df, metric, group_col='test_group', control_label='control', 
                 treatment_label='treatment', alpha=0.05):
    """
    Chi-squared test for A/B test with binary outcome.
    """
    control = df[df[group_col] == control_label]
    treatment = df[df[group_col] == treatment_label]
    
    # Create contingency table
    table = np.array([
        [control[metric].sum(), len(control) - control[metric].sum()],
        [treatment[metric].sum(), len(treatment) - treatment[metric].sum()]
    ])
    
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    
    rate_control = control[metric].mean()
    rate_treatment = treatment[metric].mean()
    lift = (rate_treatment - rate_control) / rate_control if rate_control > 0 else 0
    
    print(f"Metric: {metric}")
    print(f"Control rate: {rate_control:.4f}, Treatment rate: {rate_treatment:.4f}")
    print(f"Lift: {lift*100:.2f}%")
    print(f"Chi2: {chi2:.4f}, p-value: {p_value:.6f}")
    print(f"Significant at alpha={alpha}: {p_value < alpha}")
    print("---")
    
    return {'chi2': chi2, 'p_value': p_value, 'lift': lift, 'significant': p_value < alpha}

In [ ]:
def analyze_novelty_effect(df, metric, group_col='test_group', time_col='hrs_in_exp',
                           control_label='control', treatment_label='treatment',
                           n_buckets=10, alpha=0.05):
    """
    Analyze novelty effect by bucketing users based on hours in experiment.
    Shows how the treatment effect changes over time (hours-based).
    
    If the effect is decaying over time, this suggests a novelty effect.
    """
    df_copy = df.copy()
    df_copy['time_bucket'] = pd.qcut(df_copy[time_col], q=n_buckets, duplicates='drop')
    
    results = []
    for bucket in sorted(df_copy['time_bucket'].unique()):
        bucket_df = df_copy[df_copy['time_bucket'] == bucket]
        
        control = bucket_df[bucket_df[group_col] == control_label][metric]
        treatment = bucket_df[bucket_df[group_col] == treatment_label][metric]
        
        if len(control) > 0 and len(treatment) > 0:
            rate_control = control.mean()
            rate_treatment = treatment.mean()
            lift = (rate_treatment - rate_control) / rate_control if rate_control > 0 else 0
            
            try:
                successes = np.array([treatment.sum(), control.sum()])
                nobs = np.array([len(treatment), len(control)])
                z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')
            except:
                z_stat, p_value = np.nan, np.nan
            
            results.append({
                'time_bucket': str(bucket),
                'n_control': len(control),
                'n_treatment': len(treatment),
                'rate_control': rate_control,
                'rate_treatment': rate_treatment,
                'lift': lift,
                'lift_pct': f"{lift*100:.2f}%",
                'p_value': p_value,
                'significant': p_value < alpha if not np.isnan(p_value) else False
            })
    
    results_df = pd.DataFrame(results)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(range(len(results_df)), results_df['rate_control'], 'b-o', label='Control')
    axes[0].plot(range(len(results_df)), results_df['rate_treatment'], 'r-o', label='Treatment')
    axes[0].set_xlabel('Time Bucket (hours in experiment)')
    axes[0].set_ylabel(f'{metric} rate')
    axes[0].set_title(f'{metric} by Time in Experiment')
    axes[0].legend()
    axes[0].set_xticks(range(len(results_df)))
    axes[0].set_xticklabels(range(1, len(results_df)+1))
    
    lifts = [r['lift'] for r in results]
    axes[1].bar(range(len(lifts)), lifts, color=['green' if l > 0 else 'red' for l in lifts])
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_xlabel('Time Bucket (hours in experiment)')
    axes[1].set_ylabel('Relative Lift')
    axes[1].set_title('Treatment Lift by Time in Experiment')
    axes[1].set_xticks(range(len(lifts)))
    axes[1].set_xticklabels(range(1, len(lifts)+1))
    
    plt.tight_layout()
    plt.show()
    
    return results_df

In [ ]:
def analyze_novelty_effect_days(df, metric, group_col='test_group', time_col='days_in_exp',
                                control_label='control', treatment_label='treatment',
                                alpha=0.05):
    """
    Analyze novelty effect by bucketing users based on days in experiment.
    Shows how the treatment effect changes over time (days-based).
    
    If the effect is decaying over time, this suggests a novelty effect.
    """
    df_copy = df.copy()
    
    # If days_in_exp doesn't exist but hrs_in_exp does, compute it
    if time_col not in df_copy.columns and 'hrs_in_exp' in df_copy.columns:
        df_copy[time_col] = (df_copy['hrs_in_exp'] / 24).astype(int)
    
    days = sorted(df_copy[time_col].unique())
    
    results = []
    for day in days:
        day_df = df_copy[df_copy[time_col] == day]
        
        control = day_df[day_df[group_col] == control_label][metric]
        treatment = day_df[day_df[group_col] == treatment_label][metric]
        
        if len(control) > 0 and len(treatment) > 0:
            rate_control = control.mean()
            rate_treatment = treatment.mean()
            lift = (rate_treatment - rate_control) / rate_control if rate_control > 0 else 0
            
            try:
                successes = np.array([treatment.sum(), control.sum()])
                nobs = np.array([len(treatment), len(control)])
                z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')
            except:
                z_stat, p_value = np.nan, np.nan
            
            results.append({
                'day': day,
                'n_control': len(control),
                'n_treatment': len(treatment),
                'rate_control': rate_control,
                'rate_treatment': rate_treatment,
                'lift': lift,
                'lift_pct': f"{lift*100:.2f}%",
                'p_value': p_value,
                'significant': p_value < alpha if not np.isnan(p_value) else False
            })
    
    results_df = pd.DataFrame(results)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(results_df['day'], results_df['rate_control'], 'b-o', label='Control')
    axes[0].plot(results_df['day'], results_df['rate_treatment'], 'r-o', label='Treatment')
    axes[0].set_xlabel('Day in Experiment')
    axes[0].set_ylabel(f'{metric} rate')
    axes[0].set_title(f'{metric} by Day in Experiment')
    axes[0].legend()
    
    axes[1].bar(results_df['day'], results_df['lift'], 
               color=['green' if l > 0 else 'red' for l in results_df['lift']])
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_xlabel('Day in Experiment')
    axes[1].set_ylabel('Relative Lift')
    axes[1].set_title('Treatment Lift by Day in Experiment')
    
    plt.tight_layout()
    plt.show()
    
    return results_df

---

## Background

You now have a new PM named **Max** ("Maximus Impact") who is enthusiastic, driven, and eager to ship features fast. He means well, but sometimes pushes to call experiments early based on promising-looking numbers.

This week, you have three experiments to manage. Each one involves:
- Properly designing the experiment (sample size, duration)
- Carefully analyzing early/intermediate results
- Managing stakeholder expectations when results are tempting but not yet conclusive

Your job is to be the voice of statistical rigor while remaining constructive and collaborative.

---

## Task 1: Landing Page Photo Change

**Experiment:** Replace the baby photo on the landing page with a family photo (parents + kids together).

**Hypothesis:** A family photo will resonate better with our target audience (families with young kids) and increase bookings.

**Details:**
- **Target metric:** NBL - New Booked Listing (baseline rate ~20%)
- **Guardrail metric:** NCL - New Confirmed Listing
- **Traffic:** 6,000 users/day (split 50/50 between control and treatment)
- **MDE:** 5% relative (i.e., detect a change from 20% to 21%)
- **Launch criteria:**
  - Target must be statistically significant positive at p < 0.01
  - Guardrail must NOT be statistically significant negative at p < 0.05

### Task 1.a. Experiment Design

Calculate the required run time for this experiment.

Fill in the table below:

| Metric | Baseline Rate | MDE (relative) | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|----------------|------------------------|-------------|
| NBL    | ~20%         | 5%             | ?              | ?                      | ?           |

<details><summary>Hint</summary>

Use the sample size formula for proportions:
- Absolute MDE = baseline_rate * relative_MDE = 0.20 * 0.05 = 0.01
- For p < 0.01 significance, you need alpha = 0.01 (not the usual 0.05)
- Use `statsmodels.stats.proportion.proportion_effectsize` and `statsmodels.stats.power.NormalIndPower` or the formula: n = (Z_alpha/2 + Z_beta)^2 * (p1(1-p1) + p2(1-p2)) / (p1-p2)^2
- Days = ceil(sample_size_per_variant * 2 / daily_traffic)

</details>

In [ ]:
# Task 1.a: Calculate the required sample size and run time
# Remember: the launch criterion uses alpha = 0.01 (not 0.05)

from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

baseline_rate = # YOUR CODE HERE
mde_relative = # YOUR CODE HERE
mde_absolute = # YOUR CODE HERE
alpha = # YOUR CODE HERE (what significance level are we using for the target?)
power = 0.8
daily_traffic = # YOUR CODE HERE

# Calculate effect size
# effect_size = ...

# Calculate sample size per variant
# n_per_variant = ...

# Calculate days needed
# days_needed = ...

# Print your results
# print(f"...")

### Task 1.b. Analysis

The test has completed its full run. Load the data and perform a comprehensive analysis.

In [ ]:
df_landing = pd.read_csv('../data/dataset_landing_page.csv')
df_landing.head()

In [ ]:
# 1. Check assignment balance - are control and treatment groups roughly equal?

# YOUR CODE HERE

In [ ]:
# 2. Analyze the TARGET metric (NBL / new_booked_listing)
# Use calculate_results() with alpha=0.01 (per launch criteria)

# YOUR CODE HERE

In [ ]:
# 3. Analyze the GUARDRAIL metric (NCL / new_confirmed_listing)
# Use alpha=0.05 for the guardrail

# YOUR CODE HERE

In [ ]:
# 4. Dimensional breakdown - analyze by available dimensions
# (e.g., device, country, platform - check what columns are available)

# YOUR CODE HERE

### Task 1.c. Stakeholder Management

Max sees the ~2% relative lift in NBL and comes to your desk excited:

> *"Hey! I saw the landing page results - we got a 2% lift! That's close enough to our 5% target. Let's ship it and move on to the next thing!"*

The result is **NOT statistically significant** at p < 0.01 (per the launch criteria).

**Your task:** Write a response to Max that:
1. Acknowledges his enthusiasm (don't be dismissive!)
2. Explains why we cannot ship based on these results
3. What the options are going forward (run longer? iterate? accept and move on?)
4. Frames this constructively - what did we learn?

<details><summary>Hint</summary>

Think about:
- What does "not statistically significant" actually mean in plain language?
- What is the risk of shipping something that hasn't met significance?
- Could you run the test longer to get more power? Would that help?
- Is there a way to iterate on the treatment (different photo, different placement)?
- Frame it as "the data is telling us something useful" rather than "we failed"

</details>

**Your response to Max:**

*[Write your response here - imagine you're writing a Slack message or speaking in a 1:1]*




---

## Task 2: Recommended Price Feature

**Experiment:** Show hosts a recommended price during onboarding (based on similar listings in their area).

**Background:** This is similar to the Week 1 pricing algorithm experiment which was successful. The team believes showing price guidance earlier (during onboarding) could help new hosts set competitive prices and get booked faster.

**Details:**
- **Target metric:** NBL - New Booked Listing (baseline rate ~20%)
- **Traffic:** 4,000 users/day
- **MDE:** 5% relative
- **Launch criteria:** Target stat-sig positive at p < 0.05

### Task 2.a. Experiment Design

Calculate the required duration for this experiment.

<details><summary>Hint</summary>

Same approach as Task 1.a, but now alpha = 0.05 and daily traffic = 4,000.

</details>

In [ ]:
# Task 2.a: Calculate the required sample size and run time

baseline_rate = # YOUR CODE HERE
mde_relative = # YOUR CODE HERE
mde_absolute = # YOUR CODE HERE
alpha = # YOUR CODE HERE
power = 0.8
daily_traffic = # YOUR CODE HERE

# Calculate and print your results
# YOUR CODE HERE

### Task 2.b. Stakeholder Management - Early Results

The test has only been running for **3 days** (out of your planned duration calculated above). Max pulls you into a meeting.

> *"I know we just started this test, but I peeked at the numbers and they look AMAZING. Can we just call it early and ship?"*

Load the early results and investigate:

In [ ]:
df_price_1_early = pd.read_csv('../data/df_price_1_early.csv')
df_price_1_early.head()

In [ ]:
# Step 1: Check assignment imbalance
# Are the groups balanced?

# YOUR CODE HERE

In [ ]:
# Step 2: Show the results - they may look good!
# Use calculate_results() on the target metric

# YOUR CODE HERE

In [ ]:
# Step 3: Check for NOVELTY EFFECT
# Use analyze_novelty_effect() to see how the effect changes over time

novelty_results = analyze_novelty_effect(df_price_1_early, 'new_booked_listing')
novelty_results

**Step 4: Your response to Max**

Based on your analysis above (especially the novelty effect check), write a response to Max.

Consider:
- Is the effect real and stable, or is it decaying over time?
- What does a decaying effect mean for the long-term impact?
- Should we ship, wait, or do something else?

<details><summary>Hint</summary>

Look at the novelty effect chart:
- If the lift is highest in early time buckets and decreases in later ones, this is a classic novelty effect
- Users might be excited by the new feature initially, but the effect wears off
- If we ship based on early results, the long-term lift will likely be much smaller than what we see now
- The responsible thing is to wait for the full planned duration to see the "steady state" effect

</details>

*[Write your response to Max here]*




---

## Task 3: Recommended Price v2

**Background:** After the results from Task 2, the team iterated on the price recommendation algorithm. They improved the model and redesigned the UI to be less intrusive. A new test (v2) has been running for a few days.

### Task 3.a. Stakeholder Management - Early Results (Again!)

Max is back at your desk:

> *"OK I know you told me to wait last time, but THIS time the v2 results look even better! Can we please ship?"*

Load the early data and investigate:

In [ ]:
df_price_2_early = pd.read_csv('../data/dataset_price_v2_early.csv')
df_price_2_early.head()

In [ ]:
# 1. Check assignment imbalance

# YOUR CODE HERE

In [ ]:
# 2. Show early results

# YOUR CODE HERE

In [ ]:
# 3. Check for novelty effect
# Try analyze_novelty_effect() and/or analyze_novelty_effect_days()

# YOUR CODE HERE

**4. What do you tell Max?**

<details><summary>Hint</summary>

Even if the numbers look great, consider:
- Is there evidence of a novelty effect?
- Has the test run long enough to reach the planned sample size?
- What are the risks of shipping early vs. waiting?
- Can you give Max a specific date when you'll have conclusive results?

</details>

*[Write your response to Max here]*




### Task 3.b. Full Test Analysis

You convinced Max to wait (good job!). The full test results are now in. Time for a comprehensive analysis.

In [ ]:
df_price_2 = pd.read_csv('../data/dataset_price_v2.csv')
df_price_2.head()

In [ ]:
# Check the shape and columns
print(f"Shape: {df_price_2.shape}")
print(f"Columns: {df_price_2.columns.tolist()}")

In [ ]:
# 1. Assignment balance check

# YOUR CODE HERE

In [ ]:
# 2. Target metric analysis (NBL / new_booked_listing)

# YOUR CODE HERE

In [ ]:
# 3. Guardrail metric analysis

# YOUR CODE HERE

In [ ]:
# 4. Dimensional breakdown
# Check results across available dimensions

# YOUR CODE HERE

**5. Final Recommendation**

Based on your full analysis, what is your recommendation? Ship, iterate, or kill?

*[Write your recommendation here, supported by the data]*




### Task 3.c. (BONUS) Deep Dive

Let's dig deeper into what happened with this experiment.

#### 3.c.i. Why were early results more positive than final results?

Think about what could explain why the early/intermediate results looked more promising than the final results.

<details><summary>Hint</summary>

Consider these hypotheses:
- **Novelty effect:** Users who see the feature early are more engaged/curious, boosting short-term metrics
- **User mix:** Early adopters may be different from later users (more tech-savvy, more motivated)
- **Regression to the mean:** With small samples, random variation can create large effects that shrink with more data
- **Peeking problem:** If you check results many times, you're more likely to catch a moment when noise makes the treatment look good

</details>

*[Write your hypotheses here]*




#### 3.c.ii. Cohort Day Analysis

Use `analyze_novelty_effect_days()` on the full dataset to show how results vary by cohort day. The full dataset has an `hrs_in_exp` column that can be converted to days.

In [ ]:
# 3.c.ii: Analyze how results vary by cohort day using the full dataset
# Use analyze_novelty_effect_days() on df_price_2

# YOUR CODE HERE

**Interpretation:** What does the cohort day analysis tell you? Does it support your hypotheses from 3.c.i?

*[Write your interpretation here]*




---

## Summary

In this assignment, you practiced:

1. **Experiment design** - Calculating sample sizes and run times with different significance thresholds
2. **Analyzing early results** - Understanding why peeking at results is dangerous
3. **Detecting novelty effects** - Using time-based analysis to identify decaying treatment effects
4. **Stakeholder management** - Communicating statistical concepts to eager PMs in a constructive way

**Key takeaways:**
- Never ship based on early results unless you've pre-registered a sequential testing plan
- Always check for novelty effects, especially for UI changes
- Being a good data scientist means being a good communicator - you need to say "not yet" in a way that keeps the team motivated
- The goal is not to block shipping, but to ship the RIGHT things with confidence